# UCI student academic risk: inspect the real experiment
This optional notebook reads the original data and actual saved results. Outputs are intentionally cleared. Run `python -m src.run_pipeline` from the project root first. No synthetic data is created. All grade columns are excluded from model inputs.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
root = Path.cwd().resolve()
if not (root / 'src').exists():
    root = root.parent
sys.path.insert(0, str(root))
from src.data import load_data, split_data
from src.config import FEATURES, EXCLUDED
frame, X, y = load_data()
assert not set(EXCLUDED) & set(X.columns)
print(frame.shape, y.value_counts().sort_index())
X.head()

## Training-only exploration
The test labels are not used for feature selection, preprocessing or tuning.

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)
print('Training / test:', len(X_train), len(X_test))
X_train.assign(at_risk=y_train).groupby('failures').at_risk.agg(['count', 'mean'])

## Actual held-out metrics and training-only selection
Precision, recall and F1 are for At Risk. F2 was used for CV selection; test outcomes do not choose the default.

In [ ]:
metadata = json.loads((root / 'models/metadata.json').read_text(encoding='utf-8'))
print('Training CV selected:', metadata['selected_model'])
pd.read_csv(root / 'results/metrics_summary.csv')

In [ ]:
pd.read_csv(root / 'results/logistic_regression_cv_results.csv').sort_values('mean_test_f2', ascending=False).head()

In [ ]:
from IPython.display import Image, display
for name in ['class_distribution.png', 'roc_curves.png', 'precision_recall_curves.png',
             'logistic_regression_coefficients.png', 'decision_tree_feature_importance.png']:
    display(Image(filename=str(root / 'results/figures' / name)))

## Verify results and read limitations
UCI feature timing is not established, especially total absences. Coefficients and tree importance show associations, not causes. Probabilities are uncalibrated. This prototype must not be the sole basis for academic decisions. See `results/report.md` for the full executed report and dataset/paper citations.

In [ ]:
from src.verify import verify
verify()